<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/project_Class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# cell 1 : Libraries

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

#2:Dataset & preprocessing

In [6]:
cow='/rural_carbon_dataset1.csv'
cow_df=pd.read_csv(cow)
cow_df.head()

,Region,Month,Fertilizer_Usage_kg,Crop_Type,Crop_Area_ha,Livestock_Cows,Livestock_Pigs,Household_Energy_kWh,Renewable_Energy_Fraction,Temperature_C,Rainfall_mm,Carbon_Emission_tCO2,Year,classes
0,R0103,4,106.5,Maize,37.3,0,130,296.6,0.58,23.6,70.7,5.91,2024,Low
1,R0271,11,100.9,Wheat,47.4,95,196,408.8,0.50,24.9,275.5,21.63,2021,High
2,R0107,7,59.4,Maize,28.7,83,120,479.3,0.97,21.8,35.7,9.24,2022,Low
3,R0072,8,122.9,Rice,29.3,53,82,146.3,0.13,30.7,285.5,3.71,2024,Low
4,R0189,2,66.4,Maize,25.7,46,169,233.9,0.90,23.3,76.2,4.37,2020,Low


In [7]:
cow_df=cow_df.drop(['Region','Year','Month'], axis=1)

# 4. Feature Engineering
cow_weight = 2.0  # Approx tons of CO2e per cow/year
pig_weight = 0.3  # Approx tons of CO2e per pig/year

cow_df['Weighted_Livestock_Impact'] = (cow_df['Livestock_Cows'] * cow_weight) + (cow_df['Livestock_Pigs'] * pig_weight)
# 2. Advanced Density Feature: Connect the weighted impact with the land area
# cow_df['Livestock_Impact_per_Ha'] = cow_df['Weighted_Livestock_Impact'] / (cow_df['Crop_Area_ha'] + 1e-5)

cow_df['Fert_x_Area'] = cow_df['Fertilizer_Usage_kg'] / cow_df['Crop_Area_ha']
cow_df['Renew_x_Energy'] = cow_df['Renewable_Energy_Fraction'] * cow_df['Household_Energy_kWh']
cow_df['emission_density'] = cow_df['Carbon_Emission_tCO2'] / (cow_df['Crop_Area_ha'] + 1e-5)
cow_df['energy_intensity'] = cow_df['Household_Energy_kWh'] / (cow_df['Crop_Area_ha'] + 1e-5)
cow_df=cow_df.drop(['Livestock_Pigs','Livestock_Cows','Renewable_Energy_Fraction','Carbon_Emission_tCO2','Fertilizer_Usage_kg'], axis=1)

# print a sample of dataset after preprocessing
cow_df.head()


In [12]:
# Separate target from predictors
X = cow_df.drop(columns=['classes'])
y = cow_df['classes']

# Identify text (categorical) and numerical columns automatically
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# 3: Model

#1st Model

In [13]:
# Common in the 2 Models
preprocessor = ColumnTransformer(
    transformers=[
        # Converts text columns to 1s and 0s, ignores unseen words in test data safely
        ('text_encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        # Normalizes numerical columns to standard scales
        ('num_scale', StandardScaler(), numerical_cols)])

In [14]:
# rf_classifier = RandomForestClassifier(
#     n_estimators=100,       # Number of decision trees in the forest
#     max_depth=8,            # Maximum tree split levels to mitigate overfitting
#     min_samples_leaf=4,     # Minimum samples allowed at a leaf node
#     random_state=42,        # Standard seed for consistency
#     n_jobs=-1,              # Uses all CPU cores for parallel tree building
#     class_weight='balanced'
# )

In [15]:
# # The pipeline ensures text processing flows straight into model training
# model_pipeline = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('classifier', rf_classifier)
# ])

NameError: name 'rf_classifier' is not defined

In [22]:
# # Split into 80% train and 20% validation
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# # Training both the text-encoder and the Random Forest simultaneously
# model_pipeline.fit(X_train, y_train)

NameError: name 'model_pipeline' is not defined

# 2nd Model by Grid search

In [17]:
# 1. Initialize the base classifier (as a step in a new pipeline)
# We will build a pipeline for GridSearchCV that includes the preprocessor.
# Create a pipeline combining the preprocessor and a RandomForestClassifier
# The preprocessor is already defined as 'preprocessor'
pipe_for_grid_search = Pipeline(steps=[
    ('preprocessor', preprocessor), # Use the existing preprocessor
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1)) # A placeholder classifier
])

# 2. Define the parameter grid. Parameters need to be prefixed with the step name.
param_grid = {
    'classifier__n_estimators': [100, 300, 500],
    'classifier__max_depth': [6, 8, 12, None],
    'classifier__min_samples_leaf': [2, 4, 6],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__class_weight': ['balanced', None] # Tests handling class imbalance vs leaving it raw
}

# 3. Set up the Grid Search engine
grid_search = GridSearchCV(
    estimator=pipe_for_grid_search, # Use the pipeline as the estimator
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

# 4. Execute Grid Search on your raw training data (the pipeline will preprocess it)
grid_search.fit(X_train, y_train)

# 5. Extract and print the ultimate parameters
print("\n--- Grid Search Results ---")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_ * 100:.2f}%")

# 6. Extract the optimized model for immediate deployment
best_rf_model = grid_search.best_estimator_

Fitting 5 folds for each of 216 candidates, totalling 1080 fits

--- Grid Search Results ---
Best Parameters Found: {'classifier__class_weight': 'balanced', 'classifier__max_depth': None, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 300}
Best Cross-Validation Score: 86.50%


In [23]:
from sklearn.pipeline import Pipeline

# The pipeline ensures text processing flows straight into model training
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', best_rf_model)
])

In [25]:
from sklearn.model_selection import train_test_split

# Split into 80% train and 20% validation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# The optimized pipeline (including preprocessor and classifier) is already available as best_rf_model
# Assign best_rf_model directly to model_pipeline to avoid double preprocessing
model_pipeline = best_rf_model

# Training both the text-encoder and the Random Forest simultaneously
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text_encode',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Crop_Type']),
                                                 ('num_scale', StandardScaler(),
                                                  ['Crop_Area_ha',
                                                   'Household_Energy_kWh',
                                                   'Temperature_C',
                                                   'Rainfall_mm',
                                                   'Weighted_Livestock_Impact',
                                                   'Fert_x_Area',
                                                   'Renew_x_Energy',
                                                   'emission_density',
                                                   'energy_intensity'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_leaf=2, n_estimators=300,
                                        n_jobs=-1, random_state=42))])

In [26]:
from sklearn.metrics import classification_report, accuracy_score

# Infer predictions on raw text test features
y_pred = model_pipeline.predict(X_test)

# Print comprehensive classification metrics
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy Score: 87.17%

Classification Report:
               precision    recall  f1-score   support

        High       0.87      0.63      0.73        75
         Low       0.94      0.86      0.90       218
      Medium       0.83      0.94      0.88       293
        Safe       1.00      1.00      1.00        14

    accuracy                           0.87       600
   macro avg       0.91      0.86      0.88       600
weighted avg       0.88      0.87      0.87       600



In [27]:
# Infer predictions on training features
y_train_pred = model_pipeline.predict(X_train)

# Print comprehensive classification metrics for training set
print(f"Training Accuracy Score: {accuracy_score(y_train, y_train_pred) * 100:.2f}%")
print("\nTraining Classification Report:\n", classification_report(y_train, y_train_pred))

Training Accuracy Score: 100.00%

Training Classification Report:
               precision    recall  f1-score   support

        High       1.00      1.00      1.00       317
         Low       1.00      1.00      1.00       814
      Medium       1.00      1.00      1.00      1161
        Safe       1.00      1.00      1.00       108

    accuracy                           1.00      2400
   macro avg       1.00      1.00      1.00      2400
weighted avg       1.00      1.00      1.00      2400

